# Phase 3 — Rhetorical Role Prediction (RR) · Corporate Law (CL) + Information Technology (IT)
**Kaggle notebook · starts from *your* Phase-1 Legal LMs (all four architectures) · sentence-level classification with document context**

Everything required by §9 is an executed cell below:

| Spec item (§9) | Where |
|---|---|
| Start from the Phase-1 LM + sentence-classification head, exactly one label per sentence | §3–4 (`RRModel`: pooled sentence vectors → head), §11 submission checks |
| Sentence-only vs. context-aware modelling, position features, BiLSTM head, causal vs bidirectional, class balancing | §6 ablations (on `decoder_only`, full fine-tuning) |
| ≥ 4 substantially different experiments on **4 different Phase-1 models**, ≥ 3 different PEFT techniques, **LORMA compulsory** | §7 (`dec_rslora`, `eff_lorma`, `chk_lora`, + grid of LoRA / rsLoRA / LORMA / IA³ / Prompt-tuning / full FT on all four architectures) |
| **Mandatory knowledge distillation** (stronger/larger teacher → smaller student) | §8 (`enc_kd`, `dec_kd`: half-depth students initialised from *Phase-1* layers, logit + feature distillation from the full-FT teacher) |
| Both domains CL and IT, domain-specific vs joint | §9 |
| Error analysis of closely related roles, per-domain, position/length effects | §12 |
| §9.5 analysis questions | §13 (auto-generated numbers) + report checklist |
| Exact output format (`{doc_id: [labels…]}`, same length/order, official labels only) | §11 |

**Design in one paragraph.** A document is a list of sentences. Each sentence is tokenised with the Phase-1 tokenizer (truncated to `SENT_TOK` tokens) and terminated by `<eos>`. In **context mode** a window packs as many consecutive sentences as fit in `MAX_LEN` tokens (`<bos> s1 <eos> s2 <eos> …`), the Phase-1 backbone encodes the window, every sentence is represented by *[mean of its token states ‖ state at its `<eos>`]*, and a head predicts one label per sentence (linear, or a BiLSTM over the sentence vectors of the window = sentence-level document context). Window predictions overlap (stride = half a window) and are probability-averaged with edge down-weighting. In **sentence mode** every sentence is encoded alone. Rhetorical roles depend heavily on *where* a sentence sits in the judgment, so optional relative-position features and a learned domain embedding are added to the sentence vector (both ablated). Optional Viterbi smoothing with a label-transition matrix learned from the training labels is evaluated on dev.

**Kaggle setup:** add as Inputs (1) the `Phase_3_RR` dataset, (2) your Phase-1 outputs (`tokenizer.json`, `ckpt_tier1/*_best.pt` for all four architectures, and `ckpt_tier2/<winner>_best.pt` if you have it). Accelerator: **GPU (T4 or P100)**. Internet not required. Everything is written under `/kaggle/working/artifacts`.

**v2 (score-focused):** (1) the final models are **retrained on train + dev with several seeds and averaged** with the train-only models (§10b) — more data for the rare roles, less seed noise; (2) an optional **rare-role score-offset** extra submission (`…_bias`); (3) **earlier results are re-used**: add the previous run's *output* as an Input and every finished experiment (probabilities + weights) is copied in, so only the refit and the submissions run (~1–2 h instead of the whole grid).

**Workflow:** first run with `QUICK = True` (≈10–15 min smoke test of every cell), then `QUICK = False`. Every experiment saves its dev/test probabilities, so if the session dies you can re-run from the top and finished experiments are re-used (`FORCE_RETRAIN = False`).

## 0. Settings

In [ ]:
QUICK = False            # True = smoke test of every cell (tiny data / steps)
FORCE_RETRAIN = False    # True = ignore finished experiments saved under artifacts/
RUN_ABLATIONS = True     # context / head / position / class-balancing ablations (needed to pick the recipe)
RUN_GRID = True          # all 4 architectures x {full, LoRA, LORMA, ...} + extra PEFT techniques + 2nd KD student
RUN_DOMAIN_SPECIFIC = True   # CL-only / IT-only models vs the joint model
TIME_BUDGET_H = 6.0      # optional experiments (grid, ablations, domain study) are skipped after this many hours ...
REFIT_BUDGET_H = 10.0    # ... so that the final train+dev refit still fits into the 12 h Kaggle limit (refits stop after this)
STRICT_OVERRIDE = {"decoder_only": "dec_lorma", "efficient": "eff_rslora", "chunk_recurrent": "chk_lora", "enc_dec": "enc_kd"}   # the four you already submitted; None = pick the best valid combination automatically
STRICT_ALLOW_FULL = False    # False: the 4 "official" submissions use PEFT/KD techniques only (full FT is the upper-bound baseline)
SEED = 42
REFIT = True             # FINAL MODELS: retrain the selected recipes on train+dev (12% more data) with several seeds, average with the train-only models
REFIT_SEEDS = 2          # seeds per selected experiment (each seed = one more training run of that experiment)
USE_BIAS_TUNING = True   # also write an *extra* submission with per-role score offsets tuned on dev (helps macro-F1 on rare roles; can overfit 10 dev docs)

MAX_LEN = 512            # tokens per window (<= Phase-1 max positions)
SENT_TOK = 64            # max tokens per sentence (head+tail kept if longer)
STRIDE_SENTS = 4         # training-window grid (in sentences); eval stride = half a window
POS_DIM = 7

if QUICK:
    MAX_LEN, SENT_TOK = 256, 32
    H = dict(batch=2, sent_batch=16, eval_bs=8, epochs=1, min_steps=24, max_steps=24, log_every=8, lora_r=8,
             lr_full=5e-5, lr_peft=5e-4, lr_ia3=3e-3, lr_kd=1e-4, head_lr=1e-3, prompt_lr=1e-2, n_prompt=8,
             max_docs=10, kd_T=2.0, kd_alpha=0.7, kd_beta=0.5)
else:
    H = dict(batch=8, sent_batch=80, eval_bs=16, epochs=5, min_steps=400, max_steps=2500, log_every=100, lora_r=16,
             lr_full=5e-5, lr_peft=5e-4, lr_ia3=3e-3, lr_kd=1e-4, head_lr=1e-3, prompt_lr=1e-2, n_prompt=20,
             max_docs=0, kd_T=2.0, kd_alpha=0.7, kd_beta=0.5)
print("QUICK" if QUICK else "FULL", H)

## 1. Environment, data and Phase-1 checkpoints

In [ ]:
import importlib, os, sys, json, math, time, random, shutil, pickle, itertools, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display
for m in ["torch", "tokenizers", "numpy", "pandas", "matplotlib"]:
    mod = importlib.import_module(m); print(f"{m:12s}", getattr(mod, "__version__", "ok"))
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU {p.name} {p.total_memory/1e9:.0f} GB, cc {p.major}.{p.minor} -> autocast {'bf16' if p.major>=8 else 'fp16'}")
else:
    print("WARNING: no GPU visible")
T0 = time.time()
KAGGLE_INPUT = os.environ.get("RR_INPUT", "/kaggle/input")
WORK = os.environ.get("RR_WORK", "/kaggle/working")
os.makedirs(f"{WORK}/artifacts", exist_ok=True); os.chdir(WORK)
sys.path.insert(0, WORK)
print("datasets:", os.listdir(KAGGLE_INPUT) if os.path.isdir(KAGGLE_INPUT) else "NO INPUT DIR")

def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

NEED = {"CL_train": "CL_train.json", "IT_train": "IT_train.json", "CL_dev": "CL_dev.json", "IT_dev": "IT_dev.json",
        "CL_test": "CL_test.json", "IT_test": "IT_test.json", "vocab": "label_vocab.json"}
PATHS = {}
for k, fn in NEED.items():
    PATHS[k] = find_file(KAGGLE_INPUT, fn)
    print(("OK   " if PATHS[k] else "MISS ") + fn, "->", PATHS[k])
assert all(PATHS.values()), "Upload the Phase_3_RR folder (Train/ + Test/) as a Kaggle dataset."
PATHS["tokenizer"] = find_file(KAGGLE_INPUT, "tokenizer.json")
print("tokenizer:", PATHS["tokenizer"]); assert PATHS["tokenizer"], "Upload Phase-1 artifacts/tokenizer.json"

def find_phase1_checkpoint(arch):
    """ckpt_tier2/<arch>_best.pt (final run of the winner) beats ckpt_tier1/<arch>_best.pt. Smoke/ablation/distill ckpts are ignored."""
    wanted = {f"{arch}_best.pt", f"{arch}_final.pt", f"{arch}.pt"}
    skip = ("ckpt_smoke", "ckpt_abl", "ckpt_distill", "ckpt_rr")
    cand = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        for name in files:
            if name in wanted:
                path = os.path.join(root, name); low = path.lower().replace("\\", "/")
                if any(s in low for s in skip): continue
                score = 100 if name.endswith("_best.pt") else 0
                score += 40 if "ckpt_tier2" in low else (5 if "ckpt_tier1" in low else 0)
                cand.append((score, path))
    return max(cand, default=(0, None), key=lambda x: (x[0], x[1] or ""))[1]

ARCHS = ["decoder_only", "efficient", "chunk_recurrent", "enc_dec"]
CKPT = {a: find_phase1_checkpoint(a) for a in ARCHS}
for a in ARCHS: print(f"ckpt {a:16s} {CKPT[a]}")
assert all(CKPT.values()), f"Phase 3 needs all four Phase-1 models; missing: {[a for a,p in CKPT.items() if p is None]}"
os.makedirs("artifacts/ckpt_rr", exist_ok=True); os.makedirs("artifacts/probs", exist_ok=True)

In [ ]:
# Re-use finished experiments of an earlier run: add that notebook's *output* (or a dataset containing artifacts/probs and artifacts/ckpt_rr)
# as an Input and the matching files are copied here, so only the new steps (refit, submissions) have to run.
copied = 0
for root, dirs, files in os.walk(KAGGLE_INPUT):
    low = root.replace("\\", "/").lower()
    for sub, ext in (("probs", ".pkl"), ("ckpt_rr", ".pt")):
        if low.endswith("/" + sub):
            for fn in files:
                dst = f"artifacts/{sub}/{fn}"
                if fn.endswith(ext) and not os.path.exists(dst):
                    shutil.copy(os.path.join(root, fn), dst); copied += 1
print(f"re-used {copied} saved experiment files from earlier runs" if copied else "no earlier experiment files found in the inputs -> everything is trained from scratch")

## 2. Project files — the Phase-1 backbones (identical code to Phase 1/2, so the checkpoints load with `strict=True`)

In [ ]:
%%writefile config.py
"""Minimal ModelConfig matching Phase-1 checkpoints."""
from dataclasses import dataclass

@dataclass
class ModelConfig:
    architecture: str = "decoder_only"
    vocab_size: int = 32000
    hidden_size: int = 768
    num_layers: int = 12
    num_heads: int = 12
    num_kv_heads: int = 4
    ffn_multiplier: float = 2.75
    max_position_embeddings: int = 1024
    dropout: float = 0.1
    local_window: int = 128
    global_every: int = 4
    chunk_size: int = 128
    memory_size: int = 32

In [ ]:
%%writefile models.py
"""
Four substantially different architectures for Phase-1 legal LM pretraining.
All kept in the 50M-300M parameter range by adjusting hidden_size / num_layers
in config.py.

    1. DecoderOnlyLM        - RoPE + GQA + SwiGLU pre-norm transformer (baseline)
    2. Seq2SeqLM            - Encoder-decoder transformer (natural fit for
                               "512-word prompt -> 128-word continuation")
    3. EfficientLocalGlobalLM - local sliding-window attention with periodic
                               full-attention layers (cheap long-range mixing)
    4. ChunkRecurrentLM     - novel: encodes the prompt in fixed-size chunks,
                               compresses each chunk into a memory vector via
                               a small recurrent update, decoder cross-attends
                               to accumulated memory + local window
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# Shared primitives
# ============================================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=2.75, dropout=0.1):
        super().__init__()
        hidden = int(dim * mult)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


def rotary_freqs(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cat([freqs, freqs], dim=-1)  # (seq_len, head_dim)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, freqs):
    cos, sin = freqs.cos(), freqs.sin()
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


class GQAAttention(nn.Module):
    """Grouped-query causal self-attention with RoPE. Supports an optional
    additive attn_mask (for local-window / block patterns)."""

    def __init__(self, dim, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.drop = dropout

    def forward(self, x, freqs, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, freqs)

        rep = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=attn_mask is None,
            dropout_p=self.drop if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)


def build_local_window_mask(T, window, device):
    """Boolean mask: True = attend. Causal + only last `window` tokens.
    (Boolean instead of an additive float mask: SDPA rejects a float32 mask
    when q/k/v are bf16/fp16 under autocast.)"""
    idx = torch.arange(T, device=device)
    causal = idx[None, :] <= idx[:, None]
    local = (idx[:, None] - idx[None, :]) < window
    return (causal & local)[None, None, :, :]


def build_causal_mask(T, device):
    idx = torch.arange(T, device=device)
    return (idx[None, :] <= idx[:, None])[None, None, :, :]


# ============================================================================
# 1. Decoder-only Transformer (baseline: RoPE + GQA + SwiGLU, pre-norm)
# ============================================================================
class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, attn_mask=None):
        x = x + self.attn(self.norm1(x), freqs, attn_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        for block in self.blocks:
            x = block(x, freqs)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.9,
                 top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 eos_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            # BUG FIX: crop only the MODEL INPUT to the trained context. The old code overwrote `input_ids`
            # with its own crop, so the running sequence never grew past max_pos+1 tokens and generate.py
            # (which decodes out[0, prompt_len:]) got EMPTY text for every ctx-512 model and a truncated /
            # corrupted tail for long prompts on the ctx-1024 model.
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])
            next_logits = logits[:, -1, :].float() / max(temperature, 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if eos_token_id is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos> (would start an unrelated document)
            next_logits = _apply_repetition_penalty(next_logits, input_ids, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, input_ids, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if eos_token_id is not None and (next_token == eos_token_id).all():
                break
        return input_ids


def _apply_repetition_penalty(logits, generated, penalty):
    if penalty == 1.0:
        return logits
    for b in range(generated.size(0)):
        seen = torch.unique(generated[b])
        vals = logits[b, seen]
        logits[b, seen] = torch.where(vals < 0, vals * penalty, vals / penalty)
    return logits


def _block_repeat_ngrams(logits, generated, n):
    if n <= 0 or generated.size(1) < n:
        return logits
    B, T = generated.shape
    for b in range(B):
        seq = generated[b].tolist()
        if len(seq) < n - 1:
            continue
        prefix = tuple(seq[-(n - 1):])
        banned = set()
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.add(seq[i + n - 1])
        for tok in banned:
            logits[b, tok] = float("-inf")
    return logits


def _top_k_top_p_filter(logits, top_k, top_p):
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        kth_vals = torch.topk(logits, top_k, dim=-1).values[..., -1, None]
        logits = torch.where(logits < kth_vals, torch.full_like(logits, float("-inf")), logits)
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = F.softmax(sorted_logits, dim=-1)
        cum_probs = torch.cumsum(probs, dim=-1)
        remove = cum_probs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        sorted_logits[remove] = float("-inf")
        logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    return logits


# ============================================================================
# 2. Encoder-Decoder Transformer (natural fit: prompt -> continuation)
# ============================================================================
class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                           dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderLayerCross(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, memory, self_mask, mem_key_padding_mask=None):
        h = self.norm1(x)
        sa, _ = self.self_attn(h, h, h, attn_mask=self_mask, need_weights=False)
        x = x + sa
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory,
                                 key_padding_mask=mem_key_padding_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class Seq2SeqLM(nn.Module):
    """Encoder reads the 512-word prompt; decoder autoregressively produces
    the continuation. Uses learned positional embeddings (simplest to reason
    about across two separate sequences)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        half_layers = max(1, cfg.num_layers // 2)
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_emb = nn.Embedding(cfg.max_position_embeddings, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(half_layers)])
        self.decoder = nn.ModuleList([DecoderLayerCross(cfg) for _ in range(half_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _embed(self, ids):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        return self.drop(self.tok_emb(ids) + self.pos_emb(pos))

    def encode(self, src_ids, src_padding_mask=None):
        x = self._embed(src_ids)
        for layer in self.encoder:
            x = layer(x, key_padding_mask=src_padding_mask)
        return x

    def forward(self, src_ids, tgt_ids, labels=None, src_padding_mask=None):
        memory = self.encode(src_ids, src_padding_mask)
        x = self._embed(tgt_ids)
        T = tgt_ids.size(1)
        causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt_ids.device), diagonal=1)
        for layer in self.decoder:
            x = layer(x, memory, causal, mem_key_padding_mask=src_padding_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, eos_id, max_new_tokens, temperature=0.8,
                 top_p=0.9, top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 src_padding_mask=None):
        self.eval()
        memory = self.encode(src_ids, src_padding_mask)
        tgt = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self._embed(tgt)
            T = tgt.size(1)
            causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt.device), diagonal=1)
            h = x
            for layer in self.decoder:
                h = layer(h, memory, causal, mem_key_padding_mask=src_padding_mask)
            h = self.norm_f(h)
            next_logits = self.lm_head(h)[:, -1, :].float() / max(temperature, 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk> (<eos> ends the target)
            next_logits = _apply_repetition_penalty(next_logits, tgt, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, tgt, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tgt = torch.cat([tgt, next_token], dim=1)
            if (next_token == eos_id).all():
                break
        return tgt


# ============================================================================
# 3. Efficient / lightweight Transformer: local sliding-window attention with
#    periodic full-attention layers for global mixing.
# ============================================================================
class EfficientLocalGlobalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        local_mask = build_local_window_mask(T, self.cfg.local_window, input_ids.device)
        full_mask = build_causal_mask(T, input_ids.device)
        for i, block in enumerate(self.blocks):
            is_global = (i + 1) % self.cfg.global_every == 0
            mask = full_mask if is_global else local_mask
            x = block(x, freqs, attn_mask=mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])   # crop the input only (bug fix)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if kw.get("eos_token_id") is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos>
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# 4. Chunk-Recurrent Hybrid (novel architecture):
#    Prompt is split into fixed-size chunks. Each chunk is summarized into a
#    memory vector via a GRU-style recurrent update (memory carried across
#    chunks, cheap O(T) alternative to full attention over very long
#    contexts). The decoder then does local causal attention over recent
#    tokens PLUS cross-attention to the accumulated memory bank -- letting it
#    reference facts from far earlier in a long legal prompt (e.g. an early
#    citation or defined term) without full quadratic attention.
# ============================================================================
class ChunkEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.chunk_size = cfg.chunk_size
        self.hidden = cfg.hidden_size
        self.intra_chunk_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                        dropout=cfg.dropout, batch_first=True)
        self.norm = RMSNorm(cfg.hidden_size)
        self.summarize = nn.Linear(cfg.hidden_size, cfg.hidden_size)
        self.recurrent_update = nn.GRUCell(cfg.hidden_size, cfg.hidden_size)

    def forward(self, x):
        """x: (B, T, H) prompt embeddings. Returns memory bank (B, num_chunks, H)."""
        B, T, H = x.shape
        n_chunks = math.ceil(T / self.chunk_size)
        pad = n_chunks * self.chunk_size - T
        if pad > 0:
            x = F.pad(x, (0, 0, 0, pad))
        chunks = x.view(B, n_chunks, self.chunk_size, H)

        state = torch.zeros(B, H, device=x.device, dtype=x.dtype)
        memories = []
        for c in range(n_chunks):
            seg = chunks[:, c]                      # (B, chunk_size, H)
            h = self.norm(seg)
            attn_out, _ = self.intra_chunk_attn(h, h, h, need_weights=False)
            seg = seg + attn_out
            pooled = self.summarize(seg.mean(dim=1))  # (B, H) mean-pool the chunk
            state = self.recurrent_update(pooled, state)
            memories.append(state)
        return torch.stack(memories, dim=1)  # (B, n_chunks, H)


class ChunkRecurrentBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, memory, mem_mask=None, attn_mask=None):
        x = x + self.self_attn(self.norm1(x), freqs, attn_mask)
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory, attn_mask=mem_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class ChunkRecurrentLM(nn.Module):
    """Decoder-only interface (matches the pure-continuation task) but with
    chunk-recurrent memory injected via cross-attention at every layer."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.chunk_encoder = ChunkEncoder(cfg)
        self.blocks = nn.ModuleList([ChunkRecurrentBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        # learned "empty memory" slot so tokens in the first chunk have something to attend to
        self.null_mem = nn.Parameter(torch.zeros(1, 1, cfg.hidden_size))
        self.apply(self._init_weights)
        nn.init.normal_(self.null_mem, mean=0.0, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        memory = self.chunk_encoder(x.detach())            # (B, n_chunks, H): summary of chunk c
        n_chunks = memory.size(1)
        memory = torch.cat([self.null_mem.expand(B, -1, -1).to(memory.dtype), memory], dim=1)  # slot 0 = empty
        # CAUSALITY FIX: a token in chunk c may only read slot 0 and the summaries of chunks 0..c-1
        # (slots 1..c). The old code let every token read every chunk, including the chunk that
        # contains the very token it is asked to predict, so its val loss was artificially low.
        chunk_id = torch.arange(T, device=input_ids.device) // self.cfg.chunk_size
        slot = torch.arange(n_chunks + 1, device=input_ids.device)
        mem_mask = slot[None, :] > chunk_id[:, None]        # True = NOT allowed
        for block in self.blocks:
            x = block(x, freqs, memory, mem_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])   # crop the input only (bug fix)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if kw.get("eos_token_id") is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos>
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# Factory + param counting
# ============================================================================
def build_model(cfg):
    arch = cfg.architecture
    if arch == "decoder_only":
        return DecoderOnlyLM(cfg)
    if arch == "enc_dec":
        return Seq2SeqLM(cfg)
    if arch == "efficient":
        return EfficientLocalGlobalLM(cfg)
    if arch == "chunk_recurrent":
        return ChunkRecurrentLM(cfg)
    raise ValueError(f"Unknown architecture: {arch}")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# Checkpoint / AMP helpers shared by train / eval / generate / distill
# ============================================================================
def get_amp_dtype():
    """bf16 on Ampere+ GPUs, fp16 on T4/P100 (no native bf16), None on CPU."""
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def load_checkpoint(path, device):
    """Load a checkpoint written by train.py / distill.py. Configs are stored as plain dicts,
    so this works on torch>=2.6 where torch.load defaults to weights_only=True and cannot
    unpickle a dataclass (the old torch.load(path) calls would have failed)."""
    from config import ModelConfig
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    if isinstance(cfg, dict):
        cfg = ModelConfig(**cfg)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, cfg, ckpt

## 3. Data: label vocabulary, documents, tokenisation, windows

Robust to the usual JSON layouts (list of `{"id","text","labels"}` or `{id: {...}}`). Internal key of a document is `"<domain>|<id>"` so CL/IT ids can never collide; the original id is kept for the output files.

In [ ]:
from tokenizers import Tokenizer
from config import ModelConfig
from models import build_model, rotary_freqs, get_amp_dtype

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
amp = get_amp_dtype()
tok = Tokenizer.from_file(PATHS["tokenizer"]); tok.post_processor = None
PAD, BOS, EOS, UNK = [tok.token_to_id(t) for t in ["<pad>", "<bos>", "<eos>", "<unk>"]]
PAD = 0 if PAD is None else PAD
print("vocab", tok.get_vocab_size(), "special ids", dict(pad=PAD, bos=BOS, eos=EOS, unk=UNK), "| device", device, "| amp", amp)
assert BOS is not None and EOS is not None and SENT_TOK + 2 <= MAX_LEN

def load_vocab(path):
    v = json.load(open(path, encoding="utf-8"))
    if isinstance(v, dict) and "labels" in v: v = v["labels"]
    if isinstance(v, list):
        labels = [str(x) if x is not None else "None" for x in v]
    elif all(isinstance(x, int) for x in v.values()):
        labels = [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])]
    elif all(str(k).isdigit() for k in v):
        labels = [str(v[k]) if v[k] is not None else "None" for k in sorted(v, key=int)]
    else:
        raise ValueError(f"unrecognised label_vocab.json layout: {str(v)[:200]}")
    assert len(set(labels)) == len(labels)
    return labels

id2lab = load_vocab(PATHS["vocab"]); lab2id = {l: i for i, l in enumerate(id2lab)}; NL = len(id2lab)
print(NL, "labels:", id2lab)
DOMAINS = ["CL", "IT"]; DOM_ID = {d: i for i, d in enumerate(DOMAINS)}

@dataclass
class Doc:
    key: str; oid: str; domain: str; sents: list; labels: Optional[np.ndarray]; toks: list; n: int

def split_doc(d):
    """-> (sentences, raw_labels or None). Accepts {"text":[..],"labels":[..]}, a list of strings, a list of
    {"text"/"sentence":.., "label":..} dicts, or a list of [sentence, label] pairs."""
    TK, LK = ("text", "sentence", "sent", "sentences", "sents"), ("labels", "label", "role", "roles", "tag", "tags")
    if isinstance(d, dict):
        t = next((d[k] for k in TK if k in d), None); l = next((d[k] for k in LK if k in d), None)
        if t is None: raise ValueError(f"unrecognised document dict, keys={list(d)[:10]}")
        if t and isinstance(t[0], (dict, list, tuple)):
            return split_doc(t)
        return list(t), l
    if isinstance(d, (list, tuple)):
        if not d: return [], []
        x = d[0]
        if isinstance(x, str): return list(d), None
        if isinstance(x, dict):
            sents = [str(next((e[k] for k in TK if k in e), "")) for e in d]
            lk = next((k for k in LK if k in x), None)
            return sents, ([e.get(lk) for e in d] if lk else None)
        if isinstance(x, (list, tuple)) and len(x) >= 2:
            return [str(e[0]) for e in d], [e[1] for e in d]
    raise ValueError(f"unrecognised document structure: {str(d)[:300]}")

def parse_docs(raw, domain, labeled):
    if isinstance(raw, dict):
        if isinstance(raw.get("text"), list) and not isinstance(raw.get("text")[0:1], dict):
            items = [(str(raw.get("id", "0")), raw)]
        else:
            for w in ("data", "documents", "docs", "examples"):
                if w in raw and isinstance(raw[w], (list, dict)):
                    return parse_docs(raw[w], domain, labeled)
            items = [(str(k), v) for k, v in raw.items()]
    else:
        items = [(str(d.get("id", i)) if isinstance(d, dict) else str(i), d) for i, d in enumerate(raw)]
    docs = []
    for k, d in items:
        oid = str(d.get("id", k)) if isinstance(d, dict) else str(k)
        try:
            sents, raw_l = split_doc(d)
        except Exception as e:
            raise ValueError(f"{domain}/{k}: {e}")
        sents = [s if isinstance(s, str) else str(s) for s in sents]
        lab = None
        if labeled:
            assert raw_l is not None and len(raw_l) == len(sents), \
                f"{domain}/{oid}: labels missing or length mismatch ({None if raw_l is None else len(raw_l)} vs {len(sents)} sentences). Sample: {str(d)[:300]}"
            bad = [l for l in raw_l if (str(l) if l is not None else "None") not in lab2id]
            assert not bad, f"{domain}/{oid}: labels not in label_vocab.json: {set(map(str, bad))}"
            lab = np.array([lab2id[str(l) if l is not None else "None"] for l in raw_l], dtype=np.int64)
        docs.append(Doc(f"{domain}|{oid}", oid, domain, sents, lab, [], len(sents)))
    return docs

def tokenize_docs(docs):
    head = max(8, (SENT_TOK * 2) // 3); tail = SENT_TOK - head
    for d in docs:
        encs = tok.encode_batch([s if s.strip() else "." for s in d.sents])
        d.toks = [(e.ids if len(e.ids) <= SENT_TOK else e.ids[:head] + e.ids[-tail:]) or [UNK if UNK is not None else PAD]
                  for e in encs]
    return docs

def load_split(split):
    out = []
    for dom in DOMAINS:
        docs = parse_docs(json.load(open(PATHS[f"{dom}_{split}"], encoding="utf-8")), dom, labeled=(split != "test"))
        if QUICK and H["max_docs"]: docs = docs[:H["max_docs"]]
        out += tokenize_docs(docs)
    return out

train_list, dev_list, test_list = load_split("train"), load_split("dev"), load_split("test")
DOCS = {d.key: d for d in train_list + dev_list + test_list}
assert len(DOCS) == len(train_list) + len(dev_list) + len(test_list), "duplicate document keys"
docs_train = {d.key: d for d in train_list}; docs_dev = {d.key: d for d in dev_list}; docs_test = {d.key: d for d in test_list}

rows = []
for name, dd in [("train", docs_train), ("dev", docs_dev), ("test", docs_test)]:
    for dom in DOMAINS:
        ds = [d for d in dd.values() if d.domain == dom]
        ntok = [len(t) for d in ds for t in d.toks]
        rows.append(dict(split=name, domain=dom, docs=len(ds), sentences=sum(d.n for d in ds),
                         sent_tokens_mean=round(float(np.mean(ntok)), 1), sent_tokens_p95=int(np.percentile(ntok, 95)),
                         max_doc_sents=max(d.n for d in ds)))
display(pd.DataFrame(rows))
dist = pd.DataFrame({dom: np.bincount(np.concatenate([d.labels for d in docs_train.values() if d.domain == dom]), minlength=NL)
                     for dom in DOMAINS}, index=id2lab)
dist["total"] = dist.sum(1); dist["share_%"] = (100 * dist.total / dist.total.sum()).round(2)
print("TRAIN label distribution:"); display(dist)
ids_seen = Counter(d.oid for d in test_list); COLLISION = any(c > 1 for c in ids_seen.values())
print("CL/IT test-id collision:", COLLISION)

In [ ]:
@dataclass
class Win:
    key: str; start: int; m: int; ids: list; spans: list; eos: list; n: int; dom: int

def make_window(d, start, single):
    ids = [BOS]; spans = []; eos = []
    i = start
    while i < d.n:
        t = d.toks[i]
        if i > start and (single or len(ids) + len(t) + 1 > MAX_LEN): break
        s = len(ids); ids.extend(t); spans.append((s, len(ids))); ids.append(EOS); eos.append(len(ids) - 1); i += 1
    return Win(d.key, start, i - start, ids, spans, eos, d.n, DOM_ID[d.domain])

def eval_windows(docs, single):
    out = []
    for d in docs.values():
        s = 0
        while True:
            w = make_window(d, s, single); out.append(w)
            if w.start + w.m >= d.n: break
            s += max(1, w.m // 2)
    return out

class TrainWindows(Dataset):
    def __init__(self, docs, single):
        self.docs = list(docs.values()); self.single = single
        step = 1 if single else STRIDE_SENTS
        self.items = [(di, s) for di, d in enumerate(self.docs) for s in range(0, d.n, step)]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        di, s = self.items[i]; d = self.docs[di]
        if not self.single and s > 0:
            s = min(max(0, s + random.randint(-STRIDE_SENTS // 2, STRIDE_SENTS // 2)), d.n - 1)
        return make_window(d, s, self.single)

def pos_feats(start, m, n):
    i = np.arange(start, start + m, dtype=np.float32); r = i / max(n - 1, 1)
    return np.stack([r, r * r, np.sin(np.pi * r), np.cos(np.pi * r), np.minimum(i, 16) / 16.0,
                     np.minimum(n - 1 - i, 16) / 16.0, np.full_like(r, math.log(n + 1) / 7.0)], 1).astype(np.float32)

def collate(wins):
    B = len(wins); T = max(len(w.ids) for w in wins); S = max(w.m for w in wins)
    ids = torch.full((B, T), PAD, dtype=torch.long); attn = torch.zeros((B, T), dtype=torch.long)
    M = torch.zeros((B, S, T)); eos = torch.zeros((B, S), dtype=torch.long); sm = torch.zeros((B, S), dtype=torch.bool)
    y = torch.full((B, S), -100, dtype=torch.long); pf = torch.zeros((B, S, POS_DIM)); dom = torch.zeros(B, dtype=torch.long)
    for b, w in enumerate(wins):
        d = DOCS[w.key]; L = len(w.ids)
        ids[b, :L] = torch.tensor(w.ids); attn[b, :L] = 1
        for j, (s, e) in enumerate(w.spans):
            M[b, j, s:e] = 1.0 / (e - s); eos[b, j] = w.eos[j]
        sm[b, :w.m] = True; dom[b] = w.dom
        if d.labels is not None: y[b, :w.m] = torch.from_numpy(d.labels[w.start:w.start + w.m])
        pf[b, :w.m] = torch.from_numpy(pos_feats(w.start, w.m, d.n))
    return dict(ids=ids, attn=attn, M=M, eos=eos, sm=sm, y=y, pf=pf, dom=dom, wins=wins)

def to_dev(b):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

ex = collate([make_window(next(iter(docs_train.values())), 0, False)])
print("example context window:", ex["ids"].shape, "sentences:", int(ex["sm"].sum()), "| first labels:", [id2lab[i] for i in ex["y"][0][:6].tolist()])

## 4. Model: Phase-1 backbone → sentence vectors → head, and the PEFT / KD machinery

* **`encode_hidden`** runs the Phase-1 backbone as an *encoder* (bidirectional mask by default; `bidir=False` keeps the causal pre-training mask). For `enc_dec` the encoder half is used (the cross-attention decoder is deleted → honest parameter counts). Optional soft-prompt tokens (prompt tuning) are prepended in embedding space.
* **Sentence vector** = `[mean of sentence-token states ‖ state at the sentence's <eos>]` → LayerNorm → `+ position features` `+ domain embedding` → linear head or BiLSTM over the window's sentences.
* **PEFT** (backbone frozen, head always trained): `lora` (W + sBA), `rslora` (scale α/√r), `lorma` (multiplicative `(I + sBA)·W`, compulsory), `ia3` (learned rescaling of K, V and FFN activations), `prompt` (soft prompt tuning). *QLoRA* (needs 4-bit bitsandbytes kernels, pointless for ≤100M models), *AdaLoRA* and *Prefix tuning* are not implemented; the approaches above already exceed the spec's "≥ 4 techniques".
* **Knowledge distillation**: student = Phase-1 backbone with half of its layers (evenly spaced, still *initialised from Phase-1 weights*); loss = `α·T²·KL(student‖teacher logits) + (1-α)·CE(gold) + β·(1-cos(W·student_vec, teacher_vec))`.

In [ ]:
def sdpa_mask(attention_mask, bidirectional=True, local_window=None, global_full=False):
    """Boolean SDPA mask (True = attend). Only KEY padding is masked (masking padded queries would give all-False rows -> NaN)."""
    B, T = attention_mask.shape; dev = attention_mask.device
    mask = attention_mask[:, None, None, :].bool()
    if not bidirectional:
        mask = mask & torch.tril(torch.ones((T, T), dtype=torch.bool, device=dev))
    if local_window is not None and not global_full:
        idx = torch.arange(T, device=dev); local = (idx[:, None] - idx[None, :]).abs() < local_window
        if not bidirectional: local = local & (idx[None, :] <= idx[:, None])
        mask = mask & local
    return mask

def encode_hidden(bb, arch, ids, attn, bidir=True, prefix=None):
    """(B,T) ids -> (B,T,H) hidden states of the Phase-1 backbone used as an encoder."""
    B, T = ids.shape
    x = bb.tok_emb(ids); P = 0
    if prefix is not None:
        P = prefix.size(0); x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
        attn = torch.cat([attn.new_ones(B, P), attn], 1)
    T2 = x.size(1)
    if arch == "enc_dec":
        x = bb.drop(x + bb.pos_emb(torch.arange(T2, device=ids.device))[None])
        kpm = attn == 0
        for layer in bb.encoder: x = layer(x, key_padding_mask=kpm)
        return x[:, P:]
    x = bb.drop(x); freqs = rotary_freqs(T2, bb.head_dim, ids.device)
    if arch == "decoder_only":
        m = sdpa_mask(attn, bidir)
        for blk in bb.blocks: x = blk(x, freqs, m)
    elif arch == "efficient":
        for i, blk in enumerate(bb.blocks):
            g = (i + 1) % bb.cfg.global_every == 0
            x = blk(x, freqs, sdpa_mask(attn, bidir, None if g else bb.cfg.local_window, g))
    elif arch == "chunk_recurrent":
        mem = bb.chunk_encoder(x)
        mem = torch.cat([bb.null_mem.expand(B, -1, -1).to(mem.dtype), mem], 1)
        m = sdpa_mask(attn, True)
        for blk in bb.blocks: x = blk(x, freqs, mem, None, m)
    else:
        raise ValueError(arch)
    return bb.norm_f(x)[:, P:]

class RRModel(nn.Module):
    def __init__(self, backbone, hidden, arch, head="bilstm", bidir=True, use_pos=True, n_prompt=0, dropout=0.1, lstm_hidden=256):
        super().__init__()
        self.backbone, self.arch, self.head_kind = backbone, arch, head
        self.bidir = bool(bidir) or arch in ("enc_dec", "chunk_recurrent")   # these two are bidirectional by construction here
        D = 2 * hidden; self.D = D
        self.ln = nn.LayerNorm(D)
        self.pos_proj = nn.Linear(POS_DIM, D) if use_pos else None
        self.dom_emb = nn.Embedding(len(DOMAINS), D); nn.init.zeros_(self.dom_emb.weight)
        self.drop = nn.Dropout(dropout)
        if head == "bilstm":
            self.lstm = nn.LSTM(D, lstm_hidden, batch_first=True, bidirectional=True); feat = 2 * lstm_hidden
        else:
            feat = D
        self.classifier = nn.Linear(feat, NL)
        self.prompt = None
        if n_prompt:
            w = backbone.tok_emb.weight.detach()
            self.prompt = nn.Parameter(w[torch.randint(0, w.size(0), (n_prompt,))].clone())

    def sent_repr(self, b):
        h = encode_hidden(self.backbone, self.arch, b["ids"], b["attn"], self.bidir, self.prompt)
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            mean = torch.bmm(b["M"], h)
            last = h.gather(1, b["eos"].unsqueeze(-1).expand(-1, -1, h.size(-1)))
        return torch.cat([mean, last], -1)                       # (B,S,2H)

    def classify(self, v, b):
        v = self.ln(v)
        if self.pos_proj is not None: v = v + self.pos_proj(b["pf"])
        v = self.drop(v + self.dom_emb(b["dom"])[:, None, :])
        if self.head_kind == "bilstm":
            S = v.size(1); lens = b["sm"].sum(1).clamp(min=1).cpu()
            pk = nn.utils.rnn.pack_padded_sequence(v, lens, batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(pk)
            v, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=S)
            v = self.drop(v)
        return self.classifier(v)                                 # (B,S,NL)

    def forward(self, b, class_weights=None):
        v = self.sent_repr(b); logits = self.classify(v, b)
        loss = None
        if "y" in b and (b["y"] != -100).any():
            loss = F.cross_entropy(logits.float().reshape(-1, NL), b["y"].reshape(-1), ignore_index=-100, weight=class_weights)
        return logits, loss, v

# ---------------- PEFT ----------------
class LoRALinear(nn.Module):
    """y = Wx + s*B A x   (rsLoRA: s = alpha/sqrt(r))"""
    def __init__(self, base, r=16, alpha=32, rs=False):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad = False
        out, inn = base.weight.shape
        self.A = nn.Parameter(torch.empty(r, inn)); self.B = nn.Parameter(torch.zeros(out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); self.scale = alpha / (math.sqrt(r) if rs else r)
    def forward(self, x):
        return F.linear(x, self.base.weight, self.base.bias) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LORMA, multiplicative low-rank adaptation: W' = (I + s*B A) W  ->  y = h + s*B(A h), h = Wx (W' never materialised)."""
    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad = False
        out = base.weight.shape[0]
        self.A = nn.Parameter(torch.empty(r, out)); self.B = nn.Parameter(torch.zeros(out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); self.scale = alpha / r
    def forward(self, x):
        h = F.linear(x, self.base.weight, self.base.bias)
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class IA3Out(nn.Module):
    """(IA)^3 on keys / values: output of the frozen linear is rescaled by a learned vector (init 1)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad = False
        self.l = nn.Parameter(torch.ones(base.weight.shape[0]))
    def forward(self, x): return F.linear(x, self.base.weight, self.base.bias) * self.l

class IA3In(nn.Module):
    """(IA)^3 on the FFN hidden activation: input of the frozen down-projection is rescaled."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad = False
        self.l = nn.Parameter(torch.ones(base.weight.shape[1]))
    def forward(self, x): return F.linear(x * self.l, self.base.weight, self.base.bias)

TARGET = {"q_proj", "k_proj", "v_proj", "o_proj", "w1", "w2", "w3", "summarize"}
def inject_peft(module, kind, r=16):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            if kind == "ia3":
                if name in ("k_proj", "v_proj"): setattr(module, name, IA3Out(child)); continue
                if name == "w3": setattr(module, name, IA3In(child)); continue
            elif name in TARGET:
                if kind == "lora": setattr(module, name, LoRALinear(child, r, 2 * r)); continue
                if kind == "rslora": setattr(module, name, LoRALinear(child, r, r, rs=True)); continue
                if kind == "lorma": setattr(module, name, LoRMALinear(child, r, 2 * r)); continue
        inject_peft(child, kind, r)

def prune_backbone(bb, arch):
    """Drop what a classifier never uses: LM head (tied) and, for enc_dec, the cross-attention decoder."""
    if hasattr(bb, "lm_head"): del bb.lm_head
    if arch == "enc_dec":
        bb.decoder = nn.ModuleList(); del bb.norm_f

def shrink_backbone(bb, arch, frac=0.5):
    """KD student: keep evenly spaced layers of the PHASE-1 backbone (weights are inherited, not re-initialised)."""
    attr = "encoder" if arch == "enc_dec" else "blocks"
    lst = getattr(bb, attr); L = len(lst); depth = max(1, int(round(L * frac)))
    keep = np.linspace(0, L - 1, depth).round().astype(int).tolist()
    setattr(bb, attr, nn.ModuleList([lst[i] for i in keep]))
    return keep

def load_phase1(path, scratch=False):
    """scratch=True: same architecture/config but RANDOM weights (control experiment: are Phase-1 weights useful?)"""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    if isinstance(cfg, dict): cfg = ModelConfig(**{k: v for k, v in cfg.items() if k in ModelConfig.__dataclass_fields__})
    bb = build_model(cfg)
    if not scratch: bb.load_state_dict(ck["model"], strict=True)
    assert cfg.vocab_size == tok.get_vocab_size(), f"tokenizer vocab {tok.get_vocab_size()} != checkpoint vocab {cfg.vocab_size}"
    return bb, cfg

def count_params(model):
    n_all = sum(p.numel() for p in model.parameters()); n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_head = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone."))
    return n_all, n_tr, n_head

DEFAULT = dict(mode="ctx", head="bilstm", bidir=True, pos=True, cw_power=0.5, domains=None, r=None, teacher=None, kd_frac=0.5, scratch=False)

def fresh_model(spec):
    s = {**DEFAULT, **spec}
    bb, cfg = load_phase1(CKPT[s["arch"]], s["scratch"])
    m = s["method"]
    prune_backbone(bb, s["arch"])
    if m == "kd":
        shrink_backbone(bb, s["arch"], s["kd_frac"])
    r = s["r"] or H["lora_r"]
    if m in ("full", "kd"):
        for p in bb.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if m in ("lora", "rslora", "lorma", "ia3"): inject_peft(bb, m, r)   # m == "head" / "prompt": nothing injected
    head = s["head"] if s["mode"] == "ctx" else "linear"       # a BiLSTM over a single sentence is pointless
    return RRModel(bb, cfg.hidden_size, s["arch"], head, s["bidir"], s["pos"], H["n_prompt"] if m == "prompt" else 0)
print("model code ready")

## 5. Metrics, decoding, training loop, experiment factory

* **Macro-F1** (primary), macro-P/R as in `sklearn` defaults (labels present in gold ∪ prediction); `f1_all` additionally averages over all 13 vocabulary labels (absent labels count 0).
* **Decoding**: argmax (exactly one label per sentence) or Viterbi smoothing with a transition matrix estimated from the training label sequences (`lam` = weight of the transition term, chosen on dev).
* Each finished experiment stores dev/test probabilities in `artifacts/probs/<tag>.pkl` (and its trainable weights in `artifacts/ckpt_rr/<tag>.pt`), so a crashed session resumes where it stopped.

In [ ]:
# ---------------- metrics / decoding ----------------
def prf(gold, pred):
    cm = np.zeros((NL, NL)); np.add.at(cm, (gold, pred), 1)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    div = lambda a, b: np.divide(a, b, out=np.zeros(NL), where=b > 0)
    P, R = div(tp, tp + fp), div(tp, tp + fn); F1 = div(2 * P * R, P + R)
    present = (cm.sum(1) > 0) | (cm.sum(0) > 0)
    return dict(macro_f1=float(F1[present].mean()), macro_p=float(P[present].mean()), macro_r=float(R[present].mean()),
                f1_all=float(F1.mean()), acc=float(tp.sum() / max(cm.sum(), 1)), P=P, R=R, F1=F1, support=cm.sum(1), cm=cm)

def transition_logT(domain=None):
    C = np.ones((NL, NL))
    for d in docs_train.values():
        if domain and d.domain != domain: continue
        np.add.at(C, (d.labels[:-1], d.labels[1:]), 1)
    return np.log(C / C.sum(1, keepdims=True))
LOGT = transition_logT()

def viterbi(logp, lam):
    n = logp.shape[0]
    if lam == 0 or n == 1: return logp.argmax(1)
    delta = logp[0].copy(); back = np.zeros((n, NL), dtype=np.int64)
    for t in range(1, n):
        sc = delta[:, None] + lam * LOGT
        back[t] = sc.argmax(0); delta = sc.max(0) + logp[t]
    path = [int(delta.argmax())]
    for t in range(n - 1, 0, -1): path.append(int(back[t][path[-1]]))
    return np.array(path[::-1])

def decode_all(probs, lam=0.0, bias=None):
    b = 0.0 if bias is None else bias
    return {k: viterbi(np.log(np.clip(p, 1e-9, 1.0)) + b, lam) for k, p in probs.items()}

def score(probs, lam=0.0, domain=None, pred=None):
    pred = pred if pred is not None else decode_all({k: p for k, p in probs.items() if (domain is None or DOCS[k].domain == domain)}, lam)
    keys = [k for k in pred if DOCS[k].labels is not None and (domain is None or DOCS[k].domain == domain)]
    if not keys: return None
    return prf(np.concatenate([DOCS[k].labels for k in keys]), np.concatenate([pred[k] for k in keys]))

def avg_probs(plist, weights=None):
    weights = weights or [1.0] * len(plist); tot = float(sum(weights))
    keys = set.intersection(*[set(p) for p in plist])
    return {k: sum(w * p[k] for w, p in zip(weights, plist)) / tot for k in keys}

# ---------------- inference ----------------
@torch.no_grad()
def predict_probs(model, wins, docs):
    model.eval(); model.to(device)
    acc = {k: np.zeros((d.n, NL), dtype=np.float32) for k, d in docs.items()}; wsum = {k: np.zeros(d.n, dtype=np.float32) for k, d in docs.items()}
    use_amp = device == "cuda" and amp is not None
    order = sorted(range(len(wins)), key=lambda i: len(wins[i].ids))
    for b0 in range(0, len(order), H["eval_bs"]):
        batch = [wins[i] for i in order[b0:b0 + H["eval_bs"]]]
        b = to_dev(collate(batch))
        with torch.autocast(device_type=device, dtype=amp or torch.float16, enabled=use_amp):
            logits, _, _ = model(b)
        pr = torch.softmax(logits.float(), -1).cpu().numpy()
        for i, w in enumerate(batch):
            j = np.arange(w.m)
            left = (j + 1) if w.start > 0 else np.full(w.m, 99); right = (w.m - j) if (w.start + w.m) < w.n else np.full(w.m, 99)
            wt = np.minimum(np.minimum(left, right), 4).astype(np.float32)
            acc[w.key][w.start:w.start + w.m] += pr[i, :w.m] * wt[:, None]; wsum[w.key][w.start:w.start + w.m] += wt
    return {k: acc[k] / np.maximum(wsum[k], 1e-6)[:, None] for k in acc}

_WCACHE = {}
def get_eval_wins(split, mode, domain):
    key = (split, mode, domain)
    if key not in _WCACHE:
        base = {"dev": docs_dev, "test": docs_test}[split]
        docs = {k: d for k, d in base.items() if domain is None or d.domain == domain}
        _WCACHE[key] = (eval_windows(docs, mode == "sent"), docs)
    return _WCACHE[key]

# ---------------- optimiser / loop ----------------
def make_optimizer(model, lr, wd=0.01):
    buckets = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        kind = "bb" if n.startswith("backbone.") else ("prompt" if n.startswith("prompt") else "head")
        nd = p.ndim == 1 or n.endswith(".bias") or n.endswith(".A") or n.endswith(".B") or n.endswith(".l") or kind == "prompt"
        buckets.setdefault((kind, nd), []).append(p)
    groups = []
    for (kind, nd), ps in buckets.items():
        base = {"bb": lr, "prompt": H["prompt_lr"], "head": H["head_lr"]}[kind]
        groups.append({"params": ps, "weight_decay": 0.0 if nd else wd, "lr": base, "base_lr": base})
    return torch.optim.AdamW(groups, betas=(0.9, 0.98))

def train_loop(model, loader, loss_fn, eval_fn, lr, steps, warmup, eval_every, tag):
    model.to(device); opt = make_optimizer(model, lr)
    use_amp = device == "cuda" and amp is not None
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp == torch.float16)
    params = [p for p in model.parameters() if p.requires_grad]
    best, best_state, best_step, step, t0 = -1.0, None, 0, 0, time.time()
    model.train()
    while step < steps:
        for batch in loader:
            step += 1
            fac = step / warmup if step < warmup else 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min((step - warmup) / max(1, steps - warmup), 1.0)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * fac
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=amp or torch.float16, enabled=use_amp):
                loss, info = loss_fn(to_dev(batch))
            if not torch.isfinite(loss):
                print(f"  [{tag}] step {step}: non-finite loss, skipped", flush=True)
            else:
                if scaler.is_enabled():
                    scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); nn.utils.clip_grad_norm_(params, 1.0); opt.step()
            if step % H["log_every"] == 0:
                ex = " ".join(f"{k} {v:.3f}" for k, v in (info or {}).items())
                print(f"  [{tag}] step {step:5d}/{steps} loss {loss.item():.4f} {ex} {(time.time()-t0)/60:.1f} min", flush=True)
            if eval_fn is not None and ((step % eval_every == 0 and step >= warmup) or step == steps):
                f1 = eval_fn(model)
                print(f"  [{tag}] EVAL step {step}  dev macro-F1 = {f1:.4f}", flush=True)
                if f1 > best:
                    best, best_step = f1, step
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                model.train()
            if step >= steps: break
    if best_state is not None: model.load_state_dict(best_state)
    print(f"[{tag}] done: best dev F1 {best:.4f} @ step {best_step}, {(time.time()-t0)/60:.1f} min", flush=True)
    return best, best_step

def class_weights(domains, power, pool=None):
    if not power: return None
    cnt = np.zeros(NL)
    for d in (pool or docs_train).values():
        if domains is None or d.domain == domains: cnt += np.bincount(d.labels, minlength=NL)
    w = np.clip((cnt.max() / np.maximum(cnt, 1)) ** power, 1.0, 10.0)
    return torch.tensor(w, dtype=torch.float32, device=device)

# ---------------- bookkeeping ----------------
n_items = len(TrainWindows(docs_train, False))
STEPS = int(min(H["max_steps"], max(H["min_steps"], round(H["epochs"] * math.ceil(n_items / H["batch"])))))
EVAL_EVERY = max(10, STEPS // 6)
print(f"training items (ctx windows/epoch) {n_items} -> STEPS = {STEPS} per experiment, dev eval every {EVAL_EVERY}")

RESULTS = "artifacts/rr_results.jsonl"
if FORCE_RETRAIN and os.path.exists(RESULTS): os.remove(RESULTS)
VAL_P, TEST_P, SPEC_OF, ROW = {}, {}, {}, {}
def budget_ok(limit=None): return (time.time() - T0) / 3600 < (limit or TIME_BUDGET_H)

def register(tag, spec, vp, tp, row):
    VAL_P[tag], TEST_P[tag], SPEC_OF[tag], ROW[tag] = vp, tp, spec, row

def log_row(row):
    with open(RESULTS, "a") as f: f.write(json.dumps(row) + "\n")

def lr_for(m):
    return {"full": H["lr_full"], "kd": H["lr_kd"], "ia3": H["lr_ia3"], "prompt": H["lr_peft"]}.get(m, H["lr_peft"])

def load_trained(tag):
    m = fresh_model(SPEC_OF[tag]); blob = torch.load(f"artifacts/ckpt_rr/{tag}.pt", map_location="cpu", weights_only=False)
    res = m.load_state_dict(blob["state"], strict=False); assert not res.unexpected_keys, res.unexpected_keys
    return m

def run_experiment(tag, spec, optional=True):
    spec = {**DEFAULT, **spec, "tag": tag}
    pk = f"artifacts/probs/{tag}.pkl"
    if os.path.exists(pk) and not FORCE_RETRAIN:
        b = pickle.load(open(pk, "rb")); register(tag, b["spec"], b["val"], b["test"], b["row"]); print(f"[reuse] {tag}: dev F1 {b['row']['dev_f1']:.4f}")
        return b["row"]["dev_f1"]
    if optional and not budget_ok():
        print(f"[skip] {tag}: time budget ({TIME_BUDGET_H} h) used"); return None
    mode, dom, m = spec["mode"], spec["domains"], spec["method"]
    tr = {k: d for k, d in docs_train.items() if dom is None or d.domain == dom}
    vw, vdocs = get_eval_wins("dev", mode, dom); tw, tdocs = get_eval_wins("test", mode, dom)
    model = fresh_model(spec)
    teacher = None
    if m == "kd":
        teacher = load_trained(spec["teacher"]).to(device).eval()
        for p in teacher.parameters(): p.requires_grad = False
        model.kd_proj = nn.Linear(model.D, teacher.D, bias=False)
    n_all, n_tr, n_head = count_params(model)
    print(f"=== {tag}: {spec['arch']} / {m} / mode={mode} head={model.head_kind} bidir={model.bidir} pos={spec['pos']} cw={spec['cw_power']} "
          f"domains={dom or 'CL+IT'} | {n_all/1e6:.2f}M params, {n_tr/1e6:.3f}M trainable ({100*n_tr/n_all:.2f}%)")
    ds = TrainWindows(tr, mode == "sent")
    loader = DataLoader(ds, batch_size=H["sent_batch"] if mode == "sent" else H["batch"], shuffle=True, num_workers=0, collate_fn=collate, drop_last=len(ds) > 2 * (H["sent_batch"] if mode == "sent" else H["batch"]))
    cw = class_weights(dom, spec["cw_power"])
    def loss_fn(b):
        logits, ce, sv = model(b, cw)
        if teacher is None: return ce, None
        with torch.no_grad(): tl, _, tv = teacher(b)
        sm = b["sm"]; T = H["kd_T"]
        kd = F.kl_div(F.log_softmax(logits[sm].float() / T, -1), F.softmax(tl[sm].float() / T, -1), reduction="batchmean") * T * T
        cos = 1.0 - F.cosine_similarity(model.kd_proj(sv)[sm].float(), tv[sm].float(), dim=-1).mean()
        return H["kd_alpha"] * kd + (1 - H["kd_alpha"]) * ce + H["kd_beta"] * cos, {"ce": ce.item(), "kd": kd.item(), "cos": cos.item()}
    eval_fn = lambda mdl: (score(predict_probs(mdl, vw, vdocs)) or {"macro_f1": 0})["macro_f1"]
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    _, best_step = train_loop(model, loader, loss_fn, eval_fn, lr_for(m), STEPS, max(5, STEPS // 15), EVAL_EVERY, tag)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    if device == "cuda": torch.cuda.synchronize()
    t1 = time.time(); vp = predict_probs(model, vw, vdocs)
    if device == "cuda": torch.cuda.synchronize()
    infer_s = time.time() - t1
    tp = predict_probs(model, tw, tdocs)
    s_all = score(vp); s_cl = score(vp, domain="CL"); s_it = score(vp, domain="IT")
    n_dev_sents = sum(d.n for d in vdocs.values())
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    state = {k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items() if k in names}
    torch.save({"spec": spec, "state": state}, f"artifacts/ckpt_rr/{tag}.pt")
    row = dict(tag=tag, arch=spec["arch"], method=m, mode=mode, head=model.head_kind, bidir=model.bidir, pos=spec["pos"],
               cw_power=spec["cw_power"], domains=dom or "CL+IT", dev_f1=round(s_all["macro_f1"], 4), dev_P=round(s_all["macro_p"], 4),
               dev_R=round(s_all["macro_r"], 4), dev_acc=round(s_all["acc"], 4), f1_all13=round(s_all["f1_all"], 4),
               f1_CL=round(s_cl["macro_f1"], 4) if s_cl else None, f1_IT=round(s_it["macro_f1"], 4) if s_it else None,
               params_M=round(n_all / 1e6, 2), trainable_M=round(n_tr / 1e6, 3), adapter_M=round((n_tr - n_head) / 1e6, 3),
               trainable_pct=round(100 * n_tr / n_all, 2), steps=STEPS, best_step=best_step, train_min=round(train_min, 1),
               infer_s=round(infer_s, 2), sent_per_s=round(n_dev_sents / max(infer_s, 1e-6)), peak_gpu_GB=round(peak, 2),
               fp32_MB=round(n_all * 4 / 1e6, 1))
    register(tag, spec, vp, tp, row); log_row(row)
    pickle.dump({"spec": spec, "val": vp, "test": tp, "row": row}, open(pk, "wb"))
    print({k: row[k] for k in ["tag", "dev_f1", "dev_P", "dev_R", "f1_CL", "f1_IT", "trainable_pct", "train_min"]})
    del model, teacher, loader
    if device == "cuda": torch.cuda.empty_cache()
    return row["dev_f1"]

def alias(src, dst):
    spec = {**SPEC_OF[src], "tag": dst}; row = {**ROW[src], "tag": dst}
    shutil.copy(f"artifacts/ckpt_rr/{src}.pt", f"artifacts/ckpt_rr/{dst}.pt")
    register(dst, spec, VAL_P[src], TEST_P[src], row); log_row(row)
    pickle.dump({"spec": spec, "val": VAL_P[src], "test": TEST_P[src], "row": row}, open(f"artifacts/probs/{dst}.pkl", "wb"))

SHOW = ["tag", "arch", "method", "mode", "head", "bidir", "pos", "cw_power", "domains", "dev_f1", "dev_P", "dev_R", "f1_CL", "f1_IT",
        "trainable_pct", "params_M", "trainable_M", "train_min", "sent_per_s", "peak_gpu_GB"]
def show_results(prefix=None, cols=SHOW):
    df = pd.DataFrame(list(ROW.values()))
    if prefix: df = df[df.tag.str.startswith(prefix)]
    display(df[[c for c in cols if c in df.columns]].sort_values("dev_f1", ascending=False).reset_index(drop=True))

## 6. Context, head, position and class-balancing study (`decoder_only`, full fine-tuning)

Directly answers *"does sentence/document context help?"*. All runs: same Phase-1 backbone, same step budget, same dev set.

| tag | what changes |
|---|---|
| `abl_sent` | each sentence encoded **alone**, linear head, no position features |
| `abl_sent_pos` | sentence alone + relative-position features |
| `abl_ctx_lin` | **window context** (neighbouring sentences in the backbone), linear head, position features |
| `abl_ctx_lstm` | window context + **BiLSTM over sentence vectors** (document context), position features |
| `abl_ctx_lstm_causal` | same, but keeps the *causal* Phase-1 attention mask |
| `abl_ctx_lstm_nopos` | same as `abl_ctx_lstm` without position features |

The best one becomes the recipe (`BASE`) for every other experiment, and is re-used as `dec_full` (the full-fine-tuning upper bound and KD teacher). Then two class-balancing variants of the recipe are trained (`cb_none`: unweighted CE, `cb_p1`: full inverse-frequency weights; the default is `√` inverse frequency, clipped at 10).

In [ ]:
ABL = {
    "abl_ctx_lstm":        dict(mode="ctx",  head="bilstm", pos=True,  bidir=True),
    "abl_ctx_lin":         dict(mode="ctx",  head="linear", pos=True,  bidir=True),
    "abl_sent":            dict(mode="sent", head="linear", pos=False, bidir=True),
    "abl_sent_pos":        dict(mode="sent", head="linear", pos=True,  bidir=True),
    "abl_ctx_lstm_causal": dict(mode="ctx",  head="bilstm", pos=True,  bidir=False),
    "abl_ctx_lstm_nopos":  dict(mode="ctx",  head="bilstm", pos=False, bidir=True),
}
todo = ABL if RUN_ABLATIONS else {"abl_ctx_lstm": ABL["abl_ctx_lstm"]}
for tag, v in todo.items():
    run_experiment(tag, dict(arch="decoder_only", method="full", **v), optional=(tag != "abl_ctx_lstm"))
abl_tags = [t for t in todo if t in ROW]
BEST_ABL = max(abl_tags, key=lambda t: ROW[t]["dev_f1"])
BASE = {k: SPEC_OF[BEST_ABL][k] for k in ("mode", "head", "bidir", "pos")}
print("BEST RECIPE:", BEST_ABL, BASE, "dev F1", ROW[BEST_ABL]["dev_f1"])
if "dec_full" not in ROW: alias(BEST_ABL, "dec_full")
show_results("abl_")

In [ ]:
if RUN_ABLATIONS:
    for tag, p in [("cb_none", 0.0), ("cb_p1", 1.0)]:
        run_experiment(tag, dict(arch="decoder_only", method="full", cw_power=p, **BASE))
    df = pd.DataFrame([ROW[t] for t in ["cb_none", BEST_ABL, "cb_p1"] if t in ROW])
    df["balancing"] = df.cw_power.map({0.0: "none", 0.5: "sqrt inverse-frequency (default)", 1.0: "inverse-frequency"})
    display(df[["tag", "balancing", "dev_f1", "dev_P", "dev_R", "f1_all13", "dev_acc"]])

## 6b. Control: is the Phase-1 pre-training useful?

Same recipe and architecture but **randomly initialised** weights (`dec_scratch`) vs `dec_full`; and `dec_head` (Phase-1 backbone completely frozen, only the sentence head trained) in the grid below.

In [ ]:
run_experiment("dec_scratch", dict(arch="decoder_only", method="full", scratch=True, **BASE))
display(pd.DataFrame([ROW[t] for t in ["dec_scratch", "dec_full"] if t in ROW])[["tag", "dev_f1", "dev_P", "dev_R", "f1_CL", "f1_IT"]])

## 7. PEFT experiments on all four Phase-1 architectures (LORMA is compulsory)

Same recipe (`BASE`), same step budget, backbone frozen, new head trained. **Core runs first** (one technique per architecture — these four are the strict "4 different Phase-1 models, 4 different techniques" candidates), then the grid (`RUN_GRID`) fills in more architecture × technique cells so the final selection can pick the best valid combination.

| core run | architecture | technique |
|---|---|---|
| `dec_rslora` | decoder_only | rank-stabilised LoRA |
| `eff_lorma` | efficient (local/global attention) | **LORMA** |
| `chk_lora` | chunk_recurrent | LoRA |
| `enc_kd` | enc_dec | knowledge distillation (§8) |

*Note for the report:* for `enc_dec` and for the `cross_attn` / `intra_chunk_attn` of `chunk_recurrent`, the attention uses `nn.MultiheadAttention`, whose projections are fused tensors that can't be wrapped like `nn.Linear`; adapters therefore touch only the FFN (and `q/k/v/o` of the GQA blocks where they exist).

In [ ]:
def spec_for(arch, method, **kw):
    return dict(arch=arch, method=method, **BASE, **kw)

CORE = [("dec_rslora", "decoder_only", "rslora"), ("eff_lorma", "efficient", "lorma"), ("chk_lora", "chunk_recurrent", "lora")]
GRID = [("dec_lora", "decoder_only", "lora"), ("dec_lorma", "decoder_only", "lorma"), ("dec_ia3", "decoder_only", "ia3"),
        ("dec_prompt", "decoder_only", "prompt"), ("dec_head", "decoder_only", "head"),
        ("eff_full", "efficient", "full"), ("chk_full", "chunk_recurrent", "full"), ("enc_full", "enc_dec", "full"),
        ("eff_lora", "efficient", "lora"), ("chk_lorma", "chunk_recurrent", "lorma"), ("enc_lorma", "enc_dec", "lorma"),
        ("enc_lora", "enc_dec", "lora"), ("eff_rslora", "efficient", "rslora"), ("chk_rslora", "chunk_recurrent", "rslora"),
        ("enc_rslora", "enc_dec", "rslora"), ("eff_ia3", "efficient", "ia3"), ("chk_ia3", "chunk_recurrent", "ia3")]
for tag, arch, m in CORE:
    run_experiment(tag, spec_for(arch, m), optional=False)
show_results("dec_")

## 8. Mandatory knowledge distillation

**Teacher** = the full fine-tuned `decoder_only` model (`dec_full`: the strongest and largest model). **Student** = a Phase-1 backbone with **half of its layers** (evenly spaced layers of the Phase-1 weights, so the student also starts from the Phase-1 LM), trained with
`0.7·T²·KL(student‖teacher) + 0.3·CE(gold) + 0.5·(1 − cos(W·h_student, h_teacher))` on sentence logits (T = 2) and sentence representations (learned projection `W`). The distillation signal is architecture-agnostic, so the enc_dec student (half of the 6 encoder layers) learns from a *decoder-only* teacher. `enc_kd` is the core experiment; `dec_kd` (a half-depth decoder_only student) is an extra.

The table reports whether distillation preserves quality while shrinking the model (params, MB, inference speed).

In [ ]:
assert "dec_full" in ROW, "teacher missing"
run_experiment("enc_kd", spec_for("enc_dec", "kd", teacher="dec_full"), optional=False)
if RUN_GRID: run_experiment("dec_kd", spec_for("decoder_only", "kd", teacher="dec_full"))
kd = pd.DataFrame([ROW[t] for t in ["dec_full", "enc_kd", "dec_kd"] if t in ROW])
kd["role"] = kd.tag.map({"dec_full": "TEACHER (full FT)", "enc_kd": "student (enc_dec, half encoder)", "dec_kd": "student (decoder_only, half depth)"})
t = kd[kd.tag == "dec_full"].iloc[0]
kd["F1_retained_%"] = (100 * kd.dev_f1 / t.dev_f1).round(1); kd["size_vs_teacher_%"] = (100 * kd.params_M / t.params_M).round(1)
kd["speedup_x"] = (kd.sent_per_s / t.sent_per_s).round(2)
display(kd[["tag", "role", "dev_f1", "dev_P", "dev_R", "params_M", "fp32_MB", "F1_retained_%", "size_vs_teacher_%", "sent_per_s", "speedup_x", "train_min"]])

## 7b. Remaining grid (more architecture × technique cells; skipped automatically when the time budget is used)

In [ ]:
if RUN_GRID:
    for tag, arch, m in GRID:
        run_experiment(tag, spec_for(arch, m))
show_results()

## 9. Domain study: joint (CL+IT) vs. domain-specific models

The main experiments train **one joint model** with a learned domain embedding. Here the recipe is trained on CL only and on IT only and compared on the matching dev set against the joint model's per-domain score.

In [ ]:
if RUN_DOMAIN_SPECIFIC:
    for dom in DOMAINS:
        run_experiment(f"dom_{dom}_only", dict(arch="decoder_only", method="full", **BASE, domains=dom))
    rows = []
    for dom in DOMAINS:
        t = f"dom_{dom}_only"
        if t in ROW:
            rows.append(dict(domain=dom, joint_model_F1=ROW["dec_full"][f"f1_{dom}"], domain_specific_F1=ROW[t]["dev_f1"],
                             train_sentences_joint=sum(d.n for d in docs_train.values()),
                             train_sentences_domain=sum(d.n for d in docs_train.values() if d.domain == dom)))
    display(pd.DataFrame(rows))

## 10. Decoding (argmax vs Viterbi), ensemble, and the selection of the four official submissions

* `lam` (transition weight) is tuned per model on dev among `{0, .25, .5, 1, 1.5, 2}`; `lam = 0` is plain argmax.
* **Strict four** (spec §9.2): one experiment per Phase-1 architecture (four **different** Phase-1 models), containing **LORMA** and **KD** and at least four distinct techniques overall (⇒ ≥ 3 different PEFT techniques + KD). Among all valid combinations the one with the highest mean dev macro-F1 is chosen. Full fine-tuning is excluded unless `STRICT_ALLOW_FULL = True` (it is the upper-bound baseline, not one of the PEFT techniques listed in the spec).
* Extras (clearly labelled, *not* the spec-compliant set): best single model of any kind and a probability-averaged ensemble.

In [ ]:
LAMS = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
JOINT = [t for t in ROW if SPEC_OF[t]["domains"] is None]
LAM, rows = {}, []
for t in JOINT:
    sc = {l: score(VAL_P[t], l)["macro_f1"] for l in LAMS}; LAM[t] = max(sc, key=sc.get)
    rows.append(dict(tag=t, **{f"lam={l}": round(v, 4) for l, v in sc.items()}, best_lam=LAM[t], gain=round(sc[LAM[t]] - sc[0.0], 4)))
dec_df = pd.DataFrame(rows).sort_values("lam=0.0", ascending=False).reset_index(drop=True)
display(dec_df)
print("mean gain of Viterbi smoothing over argmax:", round(dec_df.gain.mean(), 4))

# ---- greedy probability-averaging ensemble (dev) ----
cands = [t for t in JOINT if ROW[t]["dev_f1"] > 0.05 and t != "dec_full"]
ens = [max(cands, key=lambda t: ROW[t]["dev_f1"])]; best_s = ROW[ens[0]]["dev_f1"]
while len(ens) < 6:
    trial = None
    for t in cands:
        if t in ens: continue
        s = score(avg_probs([VAL_P[x] for x in ens + [t]]))["macro_f1"]
        if s > best_s + 0.002 and (trial is None or s > trial[1]): trial = (t, s)
    if trial is None: break
    ens.append(trial[0]); best_s = trial[1]; print("ensemble +", trial[0], "->", round(best_s, 4))
ens_val = avg_probs([VAL_P[t] for t in ens]); ens_sc = {l: score(ens_val, l)["macro_f1"] for l in LAMS}
ENS_LAM = max(ens_sc, key=ens_sc.get)
print("ENSEMBLE", ens, {l: round(v, 4) for l, v in ens_sc.items()}, "-> lam", ENS_LAM)

# ---- strict four ----
EXCL = ("abl_", "cb_", "dom_", "dec_scratch")
def eligible(t): return t in JOINT and not t.startswith(EXCL) and SPEC_OF[t]["method"] != "head" and (STRICT_ALLOW_FULL or SPEC_OF[t]["method"] != "full") and t != "dec_full"
by_arch = {a: [t for t in JOINT if SPEC_OF[t]["arch"] == a and eligible(t)] for a in ARCHS}
print({a: v for a, v in by_arch.items()})
best_combo, best_val = None, -1
if STRICT_OVERRIDE and all(t in ROW for t in STRICT_OVERRIDE.values()):
    best_combo = tuple(STRICT_OVERRIDE[a] for a in ARCHS); best_val = float(np.mean([ROW[t]["dev_f1"] for t in best_combo]))
    print("using the pinned official four:", dict(zip(ARCHS, best_combo)))
elif all(by_arch.values()):
    for combo in itertools.product(*[by_arch[a] for a in ARCHS]):
        ms = [SPEC_OF[t]["method"] for t in combo]
        if "lorma" in ms and "kd" in ms and len(set(ms)) >= 4:
            v = float(np.mean([ROW[t]["dev_f1"] for t in combo]))
            if v > best_val: best_combo, best_val = combo, v
if best_combo is None:
    raise RuntimeError("No valid strict-four combination: need an experiment for each of the 4 architectures, including LORMA, KD and >=4 distinct techniques. "
                       f"Candidates per arch: {by_arch}. Re-run the missing experiments (check TIME_BUDGET_H / RUN_GRID).")
STRICT = dict(zip(ARCHS, best_combo))
display(pd.DataFrame([dict(slot=i + 1, phase1_model=a, tag=t, technique=SPEC_OF[t]["method"], dev_macro_F1=ROW[t]["dev_f1"], P=ROW[t]["dev_P"], R=ROW[t]["dev_R"],
                           trainable_pct=ROW[t]["trainable_pct"], best_lam=LAM[t]) for i, (a, t) in enumerate(STRICT.items())]))
print("mean dev macro-F1 of the strict four:", round(best_val, 4))

## 10b. Final models: refit on train + dev (several seeds) and rare-role offsets

The dev set was only used for selecting recipes. For the final predictions each selected experiment (the four official ones, the ensemble members) is **trained again on train + dev** (same recipe, same technique, `REFIT_SEEDS` seeds, step count taken from the best dev step of the original run, no early stopping) and the test probabilities of the train-only model and all refit models are **averaged**. This adds 12 % more labelled data (more examples of rare roles) and averages out seed noise.

`tune_bias` additionally searches per-role score offsets on the dev ensemble (coordinate ascent, shrunk by 0.5, only kept if it gains ≥ 0.003 dev macro-F1). It is written to a separate *extra* submission because 10 dev documents can't prove it generalises.

In [ ]:
def run_refit(tag, seed):
    rtag = f"{tag}_refit{seed}"; pk = f"artifacts/probs/{rtag}.pkl"
    if os.path.exists(pk) and not FORCE_RETRAIN:
        TEST_P[rtag] = pickle.load(open(pk, "rb"))["test"]; print(f"[reuse] {rtag}"); return rtag
    if not budget_ok(REFIT_BUDGET_H):
        print(f"[skip] {rtag}: refit time budget used"); return None
    spec = {**SPEC_OF[tag], "tag": rtag}; mode, m = spec["mode"], spec["method"]
    random.seed(SEED + 100 * seed); np.random.seed(SEED + 100 * seed); torch.manual_seed(SEED + 100 * seed)
    pool = {**docs_train, **docs_dev}
    tw, tdocs = get_eval_wins("test", mode, None)
    model = fresh_model(spec); teacher = None
    if m == "kd":
        teacher = load_trained(spec["teacher"]).to(device).eval()
        for p in teacher.parameters(): p.requires_grad = False
        model.kd_proj = nn.Linear(model.D, teacher.D, bias=False)
    steps = int(min(STEPS * 1.1, max(0.5 * STEPS, 1.1 * (ROW[tag].get("best_step") or STEPS))))
    ds = TrainWindows(pool, mode == "sent"); bs = H["sent_batch"] if mode == "sent" else H["batch"]
    loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=0, collate_fn=collate, drop_last=len(ds) > 2 * bs)
    cw = class_weights(None, spec["cw_power"], pool)
    def loss_fn(b):
        logits, ce, sv = model(b, cw)
        if teacher is None: return ce, None
        with torch.no_grad(): tl, _, tv = teacher(b)
        sm = b["sm"]; T = H["kd_T"]
        kd = F.kl_div(F.log_softmax(logits[sm].float() / T, -1), F.softmax(tl[sm].float() / T, -1), reduction="batchmean") * T * T
        cos = 1.0 - F.cosine_similarity(model.kd_proj(sv)[sm].float(), tv[sm].float(), dim=-1).mean()
        return H["kd_alpha"] * kd + (1 - H["kd_alpha"]) * ce + H["kd_beta"] * cos, None
    print(f"=== REFIT {rtag} on train+dev ({sum(d.n for d in pool.values())} sentences), {steps} steps")
    train_loop(model, loader, loss_fn, None, lr_for(m), steps, max(5, steps // 15), steps, rtag)
    tp = predict_probs(model, tw, tdocs)
    TEST_P[rtag] = tp; pickle.dump({"spec": spec, "test": tp}, open(pk, "wb"))
    del model, teacher, loader
    if device == "cuda": torch.cuda.empty_cache()
    return rtag

def tune_bias(probs, lam, grid=(-1.0, -0.5, -0.25, 0.25, 0.5, 1.0), rounds=2, min_gain=0.003):
    f = lambda b: score(probs, pred=decode_all(probs, lam, b))["macro_f1"]
    b = np.zeros(NL); base = f(b); start = base
    for _ in range(rounds):
        for c in range(NL):
            best_s, best_v = base, b[c]
            for g in grid:
                t = b.copy(); t[c] = g; sc = f(t)
                if sc > best_s + 1e-9: best_s, best_v = sc, g
            b[c] = best_v; base = best_s
    b = 0.5 * b
    gain = f(b) - start
    print(f"bias tuning: dev macro-F1 {start:.4f} -> {f(b):.4f} (kept: {gain >= min_gain}); offsets", dict(zip(id2lab, b.round(2))))
    return b if gain >= min_gain else np.zeros(NL)

FINAL_TAGS = list(dict.fromkeys(list(STRICT.values()) + ens))
if REFIT:
    for seed in range(REFIT_SEEDS):
        for tag in FINAL_TAGS: run_refit(tag, seed)
def members(tag):
    return [TEST_P[x] for x in [tag] + [f"{tag}_refit{s}" for s in range(REFIT_SEEDS)] if x in TEST_P]
print({t: len(members(t)) for t in FINAL_TAGS}, "<- test-probability sets per final experiment (1 = train-only, +1 per refit seed)")
ENS_BIAS = tune_bias(ens_val, ENS_LAM) if USE_BIAS_TUNING else np.zeros(NL)
TAG_BIAS = {t: (tune_bias(VAL_P[t], LAM[t]) if USE_BIAS_TUNING else np.zeros(NL)) for t in STRICT.values()}

## 11. Write the submissions (exact format check)

In [ ]:
RAW_TEST_LEN = {}
for dom in DOMAINS:
    raw = json.load(open(PATHS[f"{dom}_test"], encoding="utf-8"))
    for d in parse_docs(raw, dom, labeled=False): RAW_TEST_LEN[d.key] = len(d.sents)

def final_probs(tags):
    plist = [p for t in tags for p in members(t)]            # every tag contributes its train-only + refit models equally
    probs = avg_probs(plist); assert set(probs) == set(docs_test), "a member model has no test probabilities for every test document"
    return probs, len(plist)

def write_submission(name, tags, lam, bias=None):
    probs, n_models = final_probs(tags); pred = decode_all(probs, lam, bias)
    lab = {k: [id2lab[i] for i in pred[k]] for k in docs_test}
    folder = f"artifacts/submissions/{name}"; os.makedirs(folder, exist_ok=True)
    combined, per_dom = {}, {d: {} for d in DOMAINS}
    for k, d in docs_test.items():
        key = f"{d.domain}_{d.oid}" if COLLISION else d.oid
        assert len(lab[k]) == RAW_TEST_LEN[k] == d.n and all(l in lab2id for l in lab[k])
        combined[key] = lab[k]; per_dom[d.domain][d.oid] = lab[k]
    json.dump(combined, open(f"{folder}/prediction.json", "w"))
    for dom in DOMAINS: json.dump(per_dom[dom], open(f"{folder}/{dom}_prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{folder}/prediction.json", arcname="prediction.json")
    dist_ = Counter(l for v in combined.values() for l in v)
    print(f"{name:40s} tags={tags} models_averaged={n_models} lam={lam} sentences={sum(len(v) for v in combined.values())}")
    print("   predicted label counts:", dict(dist_.most_common()))
    return combined

if COLLISION: print("WARNING: CL and IT share document ids -> combined prediction.json uses '<domain>_<id>' keys; per-domain files keep the original ids.")
sfx = "_refit" if REFIT else ""
for i, (arch, t) in enumerate(STRICT.items(), 1):
    write_submission(f"official{i}_{arch}_{SPEC_OF[t]['method']}{sfx}", [t], LAM[t])
    if USE_BIAS_TUNING and TAG_BIAS[t].any():       # same model(s), plus per-role offsets tuned on this model's dev probabilities
        write_submission(f"official{i}_{arch}_{SPEC_OF[t]['method']}{sfx}_bias", [t], LAM[t], TAG_BIAS[t])
# ---- optional extras (NOT part of the spec-compliant four) ----
best_single = max([t for t in JOINT if t != "dec_full"], key=lambda t: ROW[t]["dev_f1"])
write_submission(f"extra_best_single_{best_single}{sfx}", [best_single], LAM[best_single])
write_submission(f"extra_ensemble{sfx}", ens, ENS_LAM)
if USE_BIAS_TUNING and ENS_BIAS.any():
    write_submission(f"extra_ensemble{sfx}_bias", ens, ENS_LAM, ENS_BIAS)

for folder in sorted(Path("artifacts/submissions").iterdir()):
    if folder.is_dir():
        for dom in DOMAINS:
            pj = json.load(open(folder / f"{dom}_prediction.json"))
            exp = {d.oid: d.n for d in docs_test.values() if d.domain == dom}
            assert set(pj) == set(exp) and all(len(pj[k]) == exp[k] and set(pj[k]) <= set(id2lab) for k in exp), f"{folder}/{dom}: INVALID"
print("all submission files validated: same ids, same lengths, official labels only")

## 12. Error analysis (dev set, best model)

Closely related roles, per-domain differences, and how errors depend on position in the document, document length and sentence length.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("artifacts/figs", exist_ok=True)
def show_fig(name):
    plt.tight_layout(); plt.savefig(f"artifacts/figs/{name}.png", dpi=130)
    from IPython.display import Image; plt.close(); display(Image(f"artifacts/figs/{name}.png"))

# best model on dev: single or ensemble
cand = {"ENSEMBLE": (ens_val, ENS_LAM)}; cand.update({t: (VAL_P[t], LAM[t]) for t in JOINT if not t.startswith(("abl_", "cb_")) or t == BEST_ABL})
BEST_NAME = max(cand, key=lambda n: score(*cand[n])["macro_f1"]); bp, bl = cand[BEST_NAME]
pred_dev = decode_all(bp, bl); keys = [k for k in pred_dev if DOCS[k].labels is not None]
G = np.concatenate([DOCS[k].labels for k in keys]); Pd = np.concatenate([pred_dev[k] for k in keys])
DOMV = np.concatenate([[DOCS[k].domain] * DOCS[k].n for k in keys])
RELPOS = np.concatenate([np.arange(DOCS[k].n) / max(DOCS[k].n - 1, 1) for k in keys])
DLEN = np.concatenate([[DOCS[k].n] * DOCS[k].n for k in keys]); SLEN = np.concatenate([[len(t) for t in DOCS[k].toks] for k in keys])
m = prf(G, Pd); print(f"BEST = {BEST_NAME} (lam={bl}): dev macro-F1 {m['macro_f1']:.4f}  P {m['macro_p']:.4f}  R {m['macro_r']:.4f}  acc {m['acc']:.4f}")

per = pd.DataFrame({"label": id2lab, "support": m["support"].astype(int), "P": m["P"].round(3), "R": m["R"].round(3), "F1": m["F1"].round(3)})
for dom in DOMAINS:
    md_ = DOMV == dom; mm = prf(G[md_], Pd[md_]); per[f"F1_{dom}"] = mm["F1"].round(3); per[f"n_{dom}"] = mm["support"].astype(int)
display(per.sort_values("F1"))
dm = {dom: prf(G[DOMV == dom], Pd[DOMV == dom]) for dom in DOMAINS}
print({dom: (round(v["macro_f1"], 4), round(v["acc"], 4)) for dom, v in dm.items()}, "<- macro-F1, accuracy per domain")

cm = m["cm"]; cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(9, 7.5)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NL)); ax.set_yticks(range(NL)); ax.set_xticklabels(id2lab, rotation=60, ha="right"); ax.set_yticklabels(id2lab)
for i in range(NL):
    for j in range(NL):
        if cm[i, j]: ax.text(j, i, f"{cmn[i,j]*100:.0f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .55 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("gold"); ax.set_title(f"Dev confusion (row-normalised %) - {BEST_NAME}"); fig.colorbar(im); show_fig("confusion")

pairs = [(id2lab[i], id2lab[j], int(cm[i, j]), round(100 * cmn[i, j], 1)) for i in range(NL) for j in range(NL) if i != j and cm[i, j] > 0]
print("most frequent confusions (gold -> predicted, count, % of gold class):")
display(pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:15], columns=["gold", "predicted", "count", "%_of_gold"]))

GROUPS = {"petitioner vs respondent arguments": ["ArgumentPetitioner", "ArgumentRespondent"],
          "precedent roles": ["PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled"],
          "ratio vs rulings": ["RatioOfTheDecision", "RulingByPresentCourt", "RulingByLowerCourt"],
          "fact vs issue vs lower-court ruling": ["Fact", "Issue", "RulingByLowerCourt"],
          "statute vs precedent": ["Statute", "PrecedentReliedUpon"]}
rows = []
for name, labs in GROUPS.items():
    ids = [lab2id[l] for l in labs if l in lab2id]; sub = cm[np.ix_(ids, ids)]; tot = cm[ids].sum()
    rows.append({"group": name, "gold_sentences": int(tot), "correct_%": round(100 * np.trace(sub) / max(tot, 1), 1),
                 "confused_within_group_%": round(100 * (sub.sum() - np.trace(sub)) / max(tot, 1), 1),
                 "confused_with_other_labels_%": round(100 * (tot - sub.sum()) / max(tot, 1), 1)})
display(pd.DataFrame(rows))

def bucket_table(values, bins, labels_):
    b = np.digitize(values, bins); out = []
    for i, lab in enumerate(labels_):
        mk = b == i
        if mk.sum() >= 20: out.append(dict(bucket=lab, sentences=int(mk.sum()), accuracy=round(float((G[mk] == Pd[mk]).mean()), 3), macro_F1=round(prf(G[mk], Pd[mk])["macro_f1"], 3)))
    return pd.DataFrame(out)
print("by relative position in the document:"); display(bucket_table(RELPOS, np.linspace(0.1, 0.9, 9), [f"{10*i}-{10*i+10}%" for i in range(10)]))
print("by document length (sentences):"); display(bucket_table(DLEN, [100, 200, 400, 800], ["<100", "100-200", "200-400", "400-800", ">800"]))
print("by sentence length (tokens):"); display(bucket_table(SLEN, [8, 16, 32, 64], ["<8", "8-16", "16-32", "32-64", ">=64"]))

## 13. Comparison tables and the §9.5 analysis

In [ ]:
df = pd.DataFrame([ROW[t] for t in JOINT]); df["family"] = df.method.map({"full": "full FT", "kd": "KD", "head": "frozen+head"}).fillna("PEFT")
main = df[~df.tag.str.startswith(("abl_", "cb_")) & (df.tag != "dec_scratch")]
print("== architecture x technique (dev macro-F1, joint CL+IT models) ==")
piv = main.pivot_table(index="arch", columns="method", values="dev_f1", aggfunc="max").reindex(ARCHS); display(piv)
print("== efficiency: parameters, trainable share, size, training time, inference speed ==")
display(main.sort_values("dev_f1", ascending=False)[["tag", "arch", "method", "dev_f1", "params_M", "trainable_M", "adapter_M", "trainable_pct", "fp32_MB", "train_min", "peak_gpu_GB", "sent_per_s"]].reset_index(drop=True))

fig, ax = plt.subplots(1, 2, figsize=(14, 5)); col = {a: c for a, c in zip(ARCHS, ["#4C78A8", "#F58518", "#54A24B", "#B279A2"])}
d2 = main.sort_values("dev_f1")
ax[0].barh(d2.tag, d2.dev_f1, color=[col[a] for a in d2.arch]); ax[0].set_xlabel("dev macro-F1"); ax[0].set_title("All joint experiments (colour = Phase-1 architecture)")
from matplotlib.patches import Patch
ax[0].legend(handles=[Patch(color=col[a], label=a) for a in ARCHS], fontsize=8)
for a in ARCHS:
    s = main[main.arch == a]; ax[1].scatter(s.trainable_pct, s.dev_f1, c=col[a], label=a, s=40)
    for _, r in s.iterrows(): ax[1].annotate(r.method, (r.trainable_pct, r.dev_f1), fontsize=7, xytext=(3, 2), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_xlabel("trainable parameters (% of model)"); ax[1].set_ylabel("dev macro-F1"); ax[1].set_title("Parameter-efficiency"); ax[1].legend(fontsize=8)
show_fig("experiments")

In [ ]:
L = []
def add(s): L.append(s); print(s)
g = lambda t, k="dev_f1": ROW[t][k] if t in ROW else None
add("### Auto-generated numbers for the §9.5 analysis (dev set)")
best_m = main.loc[main.dev_f1.idxmax()]
add(f"1. Best single experiment: {best_m.tag} ({best_m.arch}, {best_m.method}) macro-F1 {best_m.dev_f1}. Best PEFT/KD experiment: " +
    (lambda r: f"{r.tag} ({r.arch}, {r.method}) {r.dev_f1}" )(main[main.method != 'full'].loc[main[main.method != 'full'].dev_f1.idxmax()]))
pe = main[~main.method.isin(["full", "head"])]
for a in ARCHS:
    f = main[(main.arch == a) & (main.method == "full")]; p = pe[(pe.arch == a) & (pe.method != "kd")]
    if len(f) and len(p):
        b = p.loc[p.dev_f1.idxmax()]
        add(f"2. {a}: full FT {f.dev_f1.max():.4f} vs best PEFT {b.method} {b.dev_f1:.4f} (gap {f.dev_f1.max()-b.dev_f1:+.4f}) with {b.trainable_pct}% trainable parameters")
ctx = {t: g(t) for t in ["abl_sent", "abl_sent_pos", "abl_ctx_lin", "abl_ctx_lstm", "abl_ctx_lstm_causal", "abl_ctx_lstm_nopos"] if t in ROW}
add(f"3. Context study (decoder_only, full FT, dev macro-F1): {ctx}")
if "abl_sent" in ctx and "abl_ctx_lstm" in ctx:
    add(f"   -> window+BiLSTM context vs sentence-only: {ctx['abl_ctx_lstm']-ctx['abl_sent']:+.4f}; position features alone (sentence mode): {ctx.get('abl_sent_pos', np.nan)-ctx['abl_sent']:+.4f}")
hard = per[per.support >= 5].sort_values("F1").head(4)
add("4. Hardest roles (F1, support): " + ", ".join(f"{r.label} {r.F1:.2f} (n={r.support})" for r in hard.itertuples()))
add(f"5. Domains: CL macro-F1 {dm['CL']['macro_f1']:.4f} / acc {dm['CL']['acc']:.4f}; IT macro-F1 {dm['IT']['macro_f1']:.4f} / acc {dm['IT']['acc']:.4f}" +
    ("; domain-specific models: " + str({d: g(f'dom_{d}_only') for d in DOMAINS}) + " vs joint " + str({d: g('dec_full', f'f1_{d}') for d in DOMAINS}) if "dom_CL_only" in ROW else ""))
for t in ["enc_kd", "dec_kd"]:
    if t in ROW:
        add(f"6. KD {t}: F1 {g(t)} vs teacher {g('dec_full')} ({100*g(t)/g('dec_full'):.1f}% retained), params {g(t,'params_M')}M vs {g('dec_full','params_M')}M, "
            f"inference {g(t,'sent_per_s')} vs {g('dec_full','sent_per_s')} sentences/s")
if "dec_scratch" in ROW:
    add(f"7. Phase-1 usefulness: full FT from Phase-1 weights {g('dec_full')} vs random init {g('dec_scratch')} ({g('dec_full')-g('dec_scratch'):+.4f}); frozen backbone + head only: {g('dec_head')}")
add(f"8. Decoding: Viterbi transition smoothing changed dev macro-F1 by {dec_df.gain.mean():+.4f} on average (best lam per model). Ensemble {ens} = {ens_sc[ENS_LAM]:.4f}")
add("9. Official four: " + "; ".join(f"{a}:{t}({SPEC_OF[t]['method']}) {ROW[t]['dev_f1']}" for a, t in STRICT.items()))
open("artifacts/analysis_summary.md", "w").write("\n".join(L))
pd.DataFrame(list(ROW.values())).to_csv("artifacts/rr_results_summary.csv", index=False)
print("\nsaved artifacts/analysis_summary.md, artifacts/rr_results_summary.csv, artifacts/figs/*.png")

## 14. Report checklist (§9.5) — every item has an executed cell above

- [ ] **Which techniques work best?** → §13 architecture × technique table + `analysis_summary.md` item 1–2.
- [ ] **Can PEFT approach full adaptation?** → §13 item 2 (per architecture gap) and the parameter-efficiency scatter.
- [ ] **Does context help?** → §6 table (sentence-only vs window vs BiLSTM, causal vs bidirectional, with/without position features).
- [ ] **Class balancing** → §6 `cb_none` / default / `cb_p1`.
- [ ] **Hard roles** → §12 per-label F1, confusion matrix, related-role groups (precedent roles, petitioner vs respondent arguments, ratio vs rulings).
- [ ] **CL vs IT** → §12 per-domain F1 / label table and §9 joint vs domain-specific models.
- [ ] **KD: quality preserved, efficiency gained?** → §8 table (F1 retained %, size %, speed-up).
- [ ] **Model size / compute across experiments** → §13 efficiency table (params, trainable %, MB, minutes, peak GPU GB, sentences/s).
- [ ] **Are Phase-1 representations useful?** → §6b `dec_scratch` vs `dec_full`, frozen-backbone `dec_head`.
- [ ] **Submissions**: `artifacts/submissions/official{1..4}_*_refit.zip` (four different Phase-1 models, LORMA + KD + ≥ 3 PEFT techniques); `extra_*` are optional.
- [ ] Engineering notes worth a paragraph: sentence pooling = mean ‖ `<eos>` state; window overlap with edge-weighted probability averaging; relative-position features; domain embedding; LORMA multiplicative update; KD students inherit Phase-1 layers; no external models or weights are used anywhere.

**Download:** `artifacts/submissions/*.zip`, `artifacts/rr_results_summary.csv`, `artifacts/analysis_summary.md`, `artifacts/figs/*.png`. Do **not** download `artifacts/ckpt_rr` / `artifacts/probs` unless you want to resume.

In [ ]:
import subprocess
print(subprocess.run("du -sh artifacts/* | sort -h", shell=True, capture_output=True, text=True).stdout)
print(f"total wall-clock: {(time.time()-T0)/3600:.2f} h")